```bash
CUDA_VISIBLE_DEVICES=7 vllm serve Qwen/Qwen2.5-7B-Instruct \
    --host 0.0.0.0 \
    --port 8084 \
    --gpu-memory-utilization 0.85 \
    --enable-prefix-caching \
    --dtype bfloat16 \
    --max_model_len 32000 \
    --trust-remote-code
```

```bash
CUDA_VISIBLE_DEVICES=6 vllm serve Skywork/Skywork-o1-Open-PRM-Qwen-2.5-7B \
    --host 0.0.0.0 \
    --port 8082 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

```bash
CUDA_VISIBLE_DEVICES=5 vllm serve Qwen/Qwen2.5-Math-PRM-7B \
    --host 0.0.0.0 \
    --port 8083 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

In [3]:
# from openai import OpenAI

# OPENAI_API_KEY = "EMPTY"
# OPENAI_API_BASE = "http://localhost:{PORT}/v1"

# causal_client = OpenAI(
#     api_key=OPENAI_API_KEY,
#     base_url=OPENAI_API_BASE.format(PORT=8084),
# )
# causal_model = causal_client.models.list().data[0].id

# skywork_prm_client = OpenAI(
#     api_key=OPENAI_API_KEY,
#     base_url=OPENAI_API_BASE.format(PORT=8082),
# )
# skywork_prm_model = skywork_prm_client.models.list().data[0].id

# qwen_prm_client = OpenAI(
#     api_key=OPENAI_API_KEY,
#     base_url=OPENAI_API_BASE.format(PORT=8083),
# )
# qwen_prm_model = qwen_prm_client.models.list().data[0].id


In [100]:
# def attack(df_sample, 
#     task_text, 
#     experiment_name,
#     use_augment_question_prm=True,
#     prm_clients = [skywork_prm_client, qwen_prm_client],
#     causal_client=causal_client
# ):
#     causal_model = causal_client.models.list().data[0].id
    

#     # Apply the augmentor function to each row in the DataFrame
#     aug_results = augmentor(df_sample, task_text, client=causal_client, model=causal_model)
#     df_aug = pd.DataFrame(aug_results)

#     # Check equivalence
#     equivalence_results = equivalence_check(df_sample, df_aug, client=causal_client, model=causal_model)
#     df_aug["equivalence"] = equivalence_results["equivalence"]
#     df_aug["body_equivalence_results"] = equivalence_results["body_equivalence_results"]

#     # PRM Scorer
#     for prm_client, prm_model in zip(prm_clients, prm_models):
#         rewards = prm_scorer(questions=df["aug_problem"].tolist() if use_augment_question_prm else df["problem"].tolist(),
#                             steps=df["aug_steps"].tolist(), 
#                             client=prm_client, model=prm_model)
#         df[f"{prm_model}--aug_rewards"] = rewards
            
#     # concat the original and augmented DataFrames
#     for key in df_aug.keys():
#         df_sample[key] = df_aug[key]
    
#     # Save the DataFrame to a CSV file
#     df_sample.to_parquet(f"experiments/{experiment_name}.parquet", index=False)
#     return df_sample

In [1]:
import os
import pandas as pd
from constants.prompts_constants import (
    VERBOSE_TASK, CONSISE_TASK, EQ_TO_TEXT_TASK, CHANGE_NUMBERS_TASK, REPHRASE_TASK
)
from utils.attack_utils import chatgpt_batch_augmentor, chatgpt_batch_equivalence_checker, prm_scorer

def attack_chatgpt_batch(df,
    task_text, 
    experiment_name,
    prm_clients=None,
    run_augmentor=True,
    run_equivalence_check=True,
    run_prm_scorer=True,
    use_augment_question_prm=False,

):
    """
    Run the attack on the given dataframe using the chatgpt batch API.
    Args:
        df: The dataframe to attack. Should have the following columns:
            - problem: The problem to attack.
            - steps: The steps to attack.
        task_text: The task to use for the attack.
        experiment_name: The name of the experiment.
        prm_clients: The prm clients to use for the attack.
        run_augmentor: Whether to run the augmentor.
        run_equivalence_check: Whether to run the equivalence check.
        run_prm_scorer: WhethREPHRASE_TASKer to run the prm scorer.
        use_augment_question_prm: Whether to use the augmented question for the prm scorer.
    Returns:
        df: The dataframe with the attack results. 
            For each row, it will have the following columns:
                - problem: The problem to attack.
                - steps: The steps to attack.
                - aug_problem: The augmented question.
                - aug_steps: The augmented steps.
                - equivalence: Whether the augmented question and steps are equivalent to the original question and steps.
                - body_equivalence_results: The body of the equivalence check.
                - {prm_model}--aug_rewards: The rewards from the prm scorer for the augmented question and steps.
    """
    experiment_path = os.path.join("experiments", experiment_name)
    os.makedirs(experiment_path, exist_ok=True)
    if run_augmentor:
        df = chatgpt_batch_augmentor(df, task_text, experiment_path)
    if run_equivalence_check:
        df = chatgpt_batch_equivalence_checker(df, experiment_path)
    if run_prm_scorer:
        prm_models = [prm_client.models.list().data[0].id for prm_client in prm_clients]
        for prm_client, prm_model in zip(prm_clients, prm_models):
            rewards = prm_scorer(questions=df["aug_problem"].tolist() if use_augment_question_prm else df["problem"].tolist(),
                                steps=df["aug_steps"].tolist(), 
                                client=prm_client, model=prm_model)
            df[f"{prm_model}--aug_rewards"] = rewards
    
    df.to_parquet(os.path.join(experiment_path, "attack.parquet"), index=False)
    return df

In [2]:
df = pd.read_parquet("data/processbench.parquet")


## TODO: Filter dataset
sample_size = 100
df_sample = df.sample(sample_size, random_state=42).reset_index(drop=True)


In [14]:
aug_df = attack_chatgpt_batch(df, 
                  task_text=REPHRASE_TASK,
                  experiment_name="chatgpt_batch_rephrase",
                  prm_clients=None,
                  run_augmentor=True,
                  run_equivalence_check=True,
                  run_prm_scorer=False,
                  use_augment_question_prm=False
              )

[ChatGPT Augmentor]  2025-05-03 15:52:54.744293 validating
[ChatGPT Augmentor]  2025-05-03 15:53:55.956379 validating
[ChatGPT Augmentor]  2025-05-03 15:54:57.008177 validating
[ChatGPT Augmentor]  2025-05-03 15:55:58.443511 validating
[ChatGPT Augmentor]  2025-05-03 15:56:59.474465 validating
[ChatGPT Augmentor]  2025-05-03 15:58:00.504541 in_progress
[ChatGPT Augmentor]  2025-05-03 15:59:01.679598 in_progress
[ChatGPT Augmentor]  2025-05-03 16:00:02.776266 in_progress
[ChatGPT Augmentor]  2025-05-03 16:01:03.901095 in_progress
[ChatGPT Augmentor]  2025-05-03 16:02:04.827326 in_progress
[ChatGPT Augmentor]  2025-05-03 16:03:05.897048 finalizing
[ChatGPT Augmentor]  2025-05-03 16:04:07.029685 completed
[ChatGPT Equivalence Checker]  2025-05-03 16:06:11.508156 validating
[ChatGPT Equivalence Checker]  2025-05-03 16:07:12.665814 validating
[ChatGPT Equivalence Checker]  2025-05-03 16:08:13.774018 validating
[ChatGPT Equivalence Checker]  2025-05-03 16:09:14.836792 validating
[ChatGPT Equ

In [11]:
import pandas as pd
aug_df = pd.read_parquet("experiments/chatgpt_batch_rephrase/attack.parquet")

In [12]:
print("Problem:", aug_df["problem"].iloc[3])
print("Aug Problem:", aug_df["aug_problem"].iloc[3])
print("--------------------------------")
print("Steps:", aug_df["steps"].iloc[3])
print("Aug Steps:", aug_df["aug_steps"].iloc[3])


Problem: What is the following value when expressed as a common fraction: $$\frac{1}{2^{1}}+\frac{1}{2^{2}}+\frac{1}{2^{3}}+\cdots + \frac{1}{2^{8}}+\frac{1}{2^{9}}+\frac{1}{2^{10}}?$$
Aug Problem: Express the following as a common fraction: \(\frac{1}{2^{1}}+\frac{1}{2^{2}}+\frac{1}{2^{3}}+ ... + \frac{1}{2^{8}}+\frac{1}{2^{9}}+\frac{1}{2^{10}}\).
--------------------------------
Steps: ['To solve this problem, we can use the formula for an infinite geometric series: \\(\\sum_{i=0}^{\\infty} ar^{i} = \\frac{a}{1-r}\\). In our case, \\(a=\\frac{1}{2}\\) and \\(r=\\frac{1}{2}\\).'
 'We can rewrite the sum as: \\(\\frac{1}{2^{1}}+\\frac{1}{2^{2}}+\\frac{1}{2^{3}}+\\cdots + \\frac{1}{2^{8}}+\\frac{1}{2^{9}}+\\frac{1}{2^{10}}=\\frac{1}{2}\\left(\\frac{1}{2^{0}}+\\frac{1}{2^{1}}+\\frac{1}{2^{2}}+\\cdots + \\frac{1}{2^{8}}+\\frac{1}{2^{9}}+\\frac{1}{2^{10}}\\right)\\).'
 'Notice that the expression inside the parentheses is a finite geometric series with the first term \\(\\frac{1}{2^{0}}\\)

In [13]:
print(aug_df.body_equivalence_results.iloc[3])

<step_count>Y</step_count>
  <question_thinking>Both sets ask to express the sum \(\frac{1}{2^1} + \frac{1}{2^2} + \cdots + \frac{1}{2^{10}}\) as a common fraction, so they request the same result.</question_thinking>
  <question>Y</question>
  <step1_thinking>Both steps identify the problem as related to a geometric series and indicate the first term \(a\) and common ratio \(r\). Both descriptions and values match in identifying \(a = \frac{1}{2}\) and \(r = \frac{1}{2}\).</step1_thinking>
  <step1>Y</step1>
  <step2_thinking>Both steps show the initial sum rewritten with a factor of \(\frac{1}{2}\) taken outside the series, expressed in similar forms.</step2_thinking>
  <step2>Y</step2>
  <step3_thinking>Both steps recognize the series inside the parentheses is a finite geometric series, correctly applying the sum formula and simplifying the result correctly to \(\frac{1024-1}{512}\).</step3_thinking>
  <step3>Y</step3>
  <step4_thinking>Both steps involve multiplying the previously 

In [9]:
print(aug_df[aug_df.equivalence==False].body_equivalence_results.iloc[0])

<step_count>N</step_count>
  <question_thinking>Both questions are requesting the maximum value of the expression \(|z_1 - z_2|^2 + |z_1 - z_3|^2 + |z_2 - z_3|^2\) given \(|z_1| = 2\), \(|z_2| = 3\), and \(|z_3| = 4\).</question_thinking>
  <question>Y</question>
  <step1_thinking>Both steps explain the approach to solving the problem using a geometric analysis of the complex numbers in the complex plane.</step1_thinking>
  <step1>Y</step1>
  <step2_thinking>Both steps describe what the expression \(|z_1 - z_2|^2 + |z_1 - z_3|^2 + |z_2 - z_3|^2\) represents in terms of distances in the complex plane and introduce the idea of the triangle formed by the points.</step2_thinking>
  <step2>Y</step2>
  <step3_thinking>Both steps state the given magnitudes and the implication of the triangle inequality rule in the context of the problem.</step3_thinking>
  <step3>Y</step3>
  <step4_thinking>Both steps suggest that maximizing the sum involves positioning the points as far apart as possible wit